In [0]:
%run ./common

In [0]:

# =====================================================
# PHASE 2: CHANGED RECORD UPDATE & IDEMPOTENCY TEST
# Uses isolated QA table, not production Bronze/Silver
# =====================================================

import uuid
import json
import hashlib
from datetime import datetime, timedelta

from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType
)
from pyspark.sql import functions as F

TABLE = "github_health.ops.qa_changed_record_test"

spark.conf.set("spark.sql.session.timeZone", "UTC")

# STEP 1: Create separate QA Delta table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    record_key STRING,
    repository STRING,
    issue_state STRING,
    title STRING,
    source_updated_at TIMESTAMP,
    payload_hash STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

# Explicit schema: no inference
SCHEMA = StructType([
    StructField("record_key", StringType(), False),
    StructField("repository", StringType(), False),
    StructField("issue_state", StringType(), True),
    StructField("title", StringType(), True),
    StructField("source_updated_at", TimestampType(), True),
    StructField("payload_hash", StringType(), True),
    StructField("load_timestamp", TimestampType(), True)
])

# Unique test key: avoids altering previous QA runs
test_key = "QA_ISSUE|" + uuid.uuid4().hex

# Controlled timestamps for deterministic testing
t0 = datetime(2026, 10, 9, 9, 0, 0)
t1 = t0 + timedelta(minutes=5)
t2 = t0 + timedelta(minutes=10)
t3 = t0 + timedelta(minutes=15)

def make_source(state, source_time, processing_time):

    title = "QA test issue"

    payload = json.dumps({
        "state": state,
        "title": title,
        "source_updated_at": source_time.isoformat()
    }, sort_keys=True)

    payload_hash = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()

    rows = [(
        test_key,
        "apache/spark",
        state,
        title,
        source_time,
        payload_hash,
        processing_time
    )]

    return spark.createDataFrame(rows, SCHEMA)


def get_saved_row():
    rows = (
        spark.table(TABLE)
        .filter(F.col("record_key") == test_key)
        .collect()
    )

    assert len(rows) == 1, (
        f"Expected 1 row, found {len(rows)}"
    )

    return rows[0]


def run_merge(state, source_time, processing_time):
    source = make_source(
        state,
        source_time,
        processing_time
    )

    return merge_changed(
        spark,
        source,
        TABLE
    )


# =====================================================
# TEST 1: INSERT NEW ISSUE
# =====================================================

print("\nTEST 1: Insert OPEN issue")

inserted = run_merge("OPEN", t0, t0)
first = get_saved_row()

assert inserted == 1
assert first["issue_state"] == "OPEN"
assert first["load_timestamp"] == t0

print("PASS: New issue inserted")


# =====================================================
# TEST 2: RERUN SAME ISSUE
# =====================================================

print("\nTEST 2: Replay unchanged issue")

unchanged = run_merge("OPEN", t0, t1)
second = get_saved_row()

assert unchanged == 0
assert second["load_timestamp"] == t0

print("PASS: No duplicate or timestamp change")


# =====================================================
# TEST 3: GENUINE ISSUE UPDATE
# =====================================================

print("\nTEST 3: Change OPEN -> CLOSED")

updated = run_merge("CLOSED", t2, t2)
third = get_saved_row()

assert updated == 1
assert third["issue_state"] == "CLOSED"
assert third["load_timestamp"] == t2
assert third["payload_hash"] != first["payload_hash"]

print("PASS: Existing issue updated")
print("Old state:", first["issue_state"])
print("New state:", third["issue_state"])


# =====================================================
# TEST 4: RERUN UPDATED ISSUE
# =====================================================

print("\nTEST 4: Replay CLOSED issue")

unchanged_again = run_merge("CLOSED", t2, t3)
fourth = get_saved_row()

assert unchanged_again == 0
assert fourth["load_timestamp"] == t2

print("PASS: Updated issue remains unchanged")


# =====================================================
# TEST 5: VERIFY NO DUPLICATES
# =====================================================

count = (
    spark.table(TABLE)
    .filter(F.col("record_key") == test_key)
    .count()
)

assert count == 1

print("\nTEST 5: Duplicate check")
print("Rows for test key:", count)
print("PASS: No duplicates")


# =====================================================
# FINAL SUMMARY
# =====================================================

print("\n" + "=" * 55)
print("ALL 5 CHANGED-RECORD TESTS PASSED")
print("=" * 55)

print("Initial state       : OPEN")
print("Updated state       : CLOSED")
print("Initial insert      :", inserted)
print("Unchanged replay    :", unchanged)
print("Genuine update      :", updated)
print("Second replay       :", unchanged_again)
print("Final row count     :", count)
print("Timestamp preserved :", fourth["load_timestamp"] == t2)
